<h1> Import Libraries </h1>

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import math
from pandas.api.types import (
    is_numeric_dtype, is_bool_dtype, is_categorical_dtype,
    is_datetime64_any_dtype
)
import duckdb
import subprocess
import sys
import textwrap
import json

<h1> Global Settings </h1>

In [ ]:
pd.set_option('display.max_columns', None)
con = duckdb.connect(database='vent_ebp.duckdb')
con.execute("SET TimeZone = 'America/Chicago'")

with open("config.json", "r", encoding="utf-8-sig") as f:
    cfg = json.load(f)

clif_path = cfg["paths"]["clif"]
raw_path = cfg["paths"]["raw"]
db_path = cfg["paths"]["db"]
print(clif_path)

<h1> File paths </h1>

In [ ]:
hourly_resp_path = f'{db_path}/clif_hourly_resp_support.parquet'
patient_path = f'{clif_path}/clif_patient.parquet'
hosp_path = f'{clif_path}/clif_hospitalization.parquet'
vitals_path = f"{clif_path}/clif_vitals.parquet"
provider_path = f'{clif_path}/clif_provider.parquet'
position_path = f'{clif_path}/clif_position.parquet'
meds_continuous_path = f'{clif_path}/clif_medication_admin_continuous.parquet'
meds_intermittent_path = f'{clif_path}/clif_medication_admin_intermittent.parquet'
labs_path = f"{clif_path}/clif_labs.parquet"
ecmo_path = f"{clif_path}/clif_ecmo_mcs.parquet"
diag_path = f"{clif_path}/clif_admission_diagnosis.parquet"

<h1> Get starter vent cohort </h1>

<h2> Merge hourly vent and hourly provider data </h2>

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE hourly_data AS
WITH hourly_vent_data AS (
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_date) as date,
        recorded_hour,
        vent_episode_id,
        vent_episode_hour_seq,
        vent_episode_duration_hours,
        device_category as vent_device_category,
        mode_category as vent_mode_category,
        fio2_set,
        spo2,
        (spo2 / fio2_set) AS sf_ratio,
        peep_set,
        tidal_volume_set,
        tracheostomy,
        hospital_id,
        location_category,
        ibw,
        sex_category
    FROM '{hourly_resp_path}'
    WHERE device_category = 'imv'
        AND vent_episode_id = '1'
        AND vent_episode_duration_hours >= '24'
        --AND tidal_volume_set IS NOT NULL
        AND location_category = 'icu'
    ORDER BY hospitalizations_joined_id, recorded_date, recorded_hour
),
provider_data AS (
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_date) AS date,
        recorded_hour,
        prov_npi_shifted,
        prov_name_shifted,
        prov_spec_shifted
    FROM '{provider_path}'
    WHERE prov_npi_shifted IS NOT NULL
),
patient_data AS (
    SELECT
        hospitalizations_joined_id,
        ANY_VALUE(patient_id) as mdm_link_id
    FROM '{hosp_path}'
    GROUP BY hospitalizations_joined_id
)
SELECT
    v.*,
    p.prov_npi_shifted,
    p.prov_name_shifted,
    p.prov_spec_shifted,
    pat.mdm_link_id
FROM hourly_vent_data as v
LEFT JOIN patient_data as pat USING(hospitalizations_joined_id)
INNER JOIN provider_data AS p USING(hospitalizations_joined_id, date, recorded_hour);

SELECT * FROM hourly_data
""")

In [ ]:
fix provider -- should match new lpv

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE data AS
-- 1: Get fifth vent hour
WITH fifth_hour AS (
    SELECT
        hospitalizations_joined_id,
        MIN(vent_episode_hour_seq) + 4 as vent_episode_hour_seq
    FROM hourly_data
    GROUP BY hospitalizations_joined_id
),

-- 2: Join with main data to get provider info
physician_df AS (
    SELECT d.*
    FROM hourly_data d
    INNER JOIN fifth_hour USING (hospitalizations_joined_id, vent_episode_hour_seq)
),

-- 3: Get providers who started MV on 5th hour with ≥25 cases
provider_counts AS (
    SELECT
        prov_npi_shifted,
        COUNT(*) AS count
    FROM physician_df
    GROUP BY prov_npi_shifted
    HAVING count >= 25
),
        
-- 4: Keep firstTime only if provider meets criteria
firstTime AS (
    SELECT p.*
    FROM physician_df p
    INNER JOIN provider_counts USING(prov_npi_shifted)
),
    
-- 5: Get all 2pm values
data_2pm AS (
    SELECT hd.*
    FROM hourly_data hd
    INNER JOIN provider_counts pc USING (prov_npi_shifted)
    WHERE hd.recorded_hour = 14.0
),
        
-- 6: Start with data_2pm, replace with firstTime records where there's a match
initial_data AS (
    SELECT * FROM firstTime
    UNION ALL
    SELECT d.*
    FROM data_2pm d
    LEFT JOIN firstTime f
        ON f.hospitalizations_joined_id = d.hospitalizations_joined_id
        AND f.date = d.date
    WHERE f.hospitalizations_joined_id IS NULL
)

SELECT *
FROM initial_data
ORDER BY hospitalizations_joined_id, date, recorded_hour;
SELECT * FROM data
""")

In [ ]:
con.sql(f"SELECT COUNT(Distinct prov_npi_shifted) from data")

<h1> Patient </h1>

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW patient AS
    SELECT DISTINCT ON (mdm_link_id)
        mdm_link_id,
        sex_category,
        language_category,
        race_category,
        ethnicity_category,
        death_dttm,
        DATE(death_dttm) as death_date,
        birth_date
    FROM '{patient_path}';
SELECT * FROM patient
""")

<h1> Height </h1>

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW height AS
    SELECT
        h.patient_id as mdm_link_id,
        ANY_VALUE(v.vital_value) as height_cm
    FROM '{vitals_path}' v
    INNER JOIN '{hosp_path}' h USING (hospitalizations_joined_id)
    WHERE v.vital_category = 'height_cm'
    GROUP BY mdm_link_id;
SELECT * FROM height
""")

<h1> Hospitalization </h1>

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW hospitalization AS
WITH base AS (
    SELECT
        hospitalizations_joined_id,
        MIN(admission_dttm) AS admission_dttm,
        MAX(discharge_dttm) AS discharge_dttm,
        ANY_VALUE(age_at_admission) AS age,
        ANY_VALUE(elix_vw) AS elix_vw
    FROM '{hosp_path}'
    GROUP BY hospitalizations_joined_id
)
SELECT
    *,
    DATE(admission_dttm) AS admission_date,
    DATE(discharge_dttm) AS discharge_date,
    YEAR(admission_dttm) AS admit_year,
    DATEDIFF('minute', admission_dttm, discharge_dttm) AS hosp_los_min,
    DATEDIFF('minute', admission_dttm, discharge_dttm) / 60.0    AS hosp_los_hr,
    DATEDIFF('minute', admission_dttm, discharge_dttm) / 1440.0  AS hosp_los_day
FROM base;
SELECT * FROM hospitalization
""")

<h1> Vent data daily aggregate </h1>

In [ ]:
# Get vent info
con.sql(f"""
CREATE OR REPLACE TEMP VIEW vent_daily AS
    SELECT 
        hospitalizations_joined_id,
        DATE(recorded_date) AS date,
        COUNT(recorded_hour) AS daily_hours_on_vent,
        MIN(fio2_set) as vent_fio2_min,
        MAX(fio2_set) as vent_fio2_max,
        MIN(peep_set) AS vent_peep_min,
        MAX(peep_set) AS vent_peep_max,
    FROM '{hourly_resp_path}'
    WHERE device_category = 'imv'
        AND vent_episode_id = '1'
        AND vent_episode_duration_hours >= '24'
        AND location_category = 'icu'
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM vent_daily;
""")

<h1> Prone </h1>

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW prone AS
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_dttm) as date,
        TRUE as prone_position
    FROM '{position_path}'
    WHERE position_category = 'prone'
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM prone
""")

<h1> SUP </h1>

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW sup AS
WITH continuous AS (
    SELECT
        hospitalizations_joined_id,
        DATE(admin_dttm) as date,
        TRUE as sup
    FROM '{meds_continuous_path}'
    WHERE med_category IN (
        'omeprazole',
        'pantoprazole',
        'esomeprazole',
        'lansoprazole',
        'rabeprazole',
        'famotidine',
        'ranitidine',
        'nizatidine',
        'cimetidine'
    )
),
intermittent AS (
    SELECT
        hospitalizations_joined_id,
        DATE(admin_dttm) as date,
        TRUE as sup
    FROM '{meds_intermittent_path}'
    WHERE med_category IN (
        'omeprazole',
        'pantoprazole',
        'esomeprazole',
        'lansoprazole',
        'rabeprazole',
        'famotidine',
        'ranitidine',
        'nizatidine',
        'cimetidine'
    )
),
combined AS (
    SELECT * FROM continuous
    UNION ALL
    SELECT * FROM intermittent
)
SELECT *
FROM combined
GROUP BY hospitalizations_joined_id, date, sup;
SELECT * FROM sup
""")

<h1> Glucose Control (needs work)</h1>

In [ ]:
con.sql(f"""
    SELECT *
    FROM '{labs_path}'
    WHERE lab_category ILIKE '%glucose%'
""")

In [ ]:
# test = con.sql(f"""
#     SELECT
#         data.*
#     FROM 'Z:/DataStageData/Eddington/LTVV/260309/LPV_final_data.parquet' data
#     INNER JOIN ecmo USING(hospitalizations_joined_id, date)
# """).df()
# test['ards_eligible'].value_counts()

<h1> ECMO </h1>

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW ecmo AS
    SELECT
        REGEXP_REPLACE(hospitalization_id, '_[^_]+$', '') as hospitalizations_joined_id,
        DATE(recorded_dttm) as date,
        TRUE as ecmo
    FROM '{ecmo_path}'
    WHERE mcs_group = 'ecmo'
    GROUP BY hospitalizations_joined_id, date;
""")

ecmo_count = con.sql("SELECT COUNT(*) FROM ecmo").fetchone()[0]
if ecmo_count == 0:
    raise ValueError("ecmo view has 0 rows — check ecmo_path and mcs_group filter")
else:
    print(f"ecmo: {ecmo_count} rows")

con.sql("SELECT * FROM ecmo")

<h1> Acidosis (pH < 7.2) </h1>

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW ph AS
    SELECT
        hospitalizations_joined_id,
        DATE(lab_collect_dttm) as date,
        MIN(lab_value_numeric) as ph_min_arterial_or_venous
    FROM '{labs_path}'
    WHERE lab_category IN ('ph_arterial', 'ph_venous')
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM ph
""")

<h1> SPO2 daily min </h1>

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW spo2_min AS
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_dttm) as date,
        MIN(vital_value) as spo2_min
    FROM '{vitals_path}'
    WHERE vital_category = 'spo2'
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM spo2_min
""")

<h1> PCO2 </h1>

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW pco2 AS
    SELECT
        hospitalizations_joined_id,
        DATE(lab_collect_dttm) as date,
        MIN(lab_value_numeric) as pco2_min_arterial_or_venous
    FROM '{labs_path}'
    WHERE lab_category IN ('pco2_venous', 'pco2_arterial')
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM pco2
""")

<h1> Paralysis </h1>

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW paralytic_meds AS
WITH continuous AS (
    SELECT
        hospitalizations_joined_id,
        DATE(admin_dttm) as date,
        TRUE as paralytic_meds
    FROM '{meds_continuous_path}'
    WHERE med_group = 'paralytics'
        AND med_dose != '0'
),
intermittent AS (
    SELECT
        hospitalizations_joined_id,
        DATE(admin_dttm) as date,
        TRUE as paralytic_meds
    FROM '{meds_intermittent_path}'
    WHERE med_group = 'paralytics'
        AND med_dose != '0'
),
combined AS (
    SELECT * FROM continuous
    UNION ALL
    SELECT * FROM intermittent
)
SELECT *
FROM combined
GROUP BY hospitalizations_joined_id, date, paralytic_meds;
SELECT * FROM paralytic_meds
""")

<h1> Diagnosis codes for alcohol withdrawal, seizure, or icp </h1>

In [ ]:
# con.sql(f"""
# CREATE OR REPLACE VIEW diagnosis_codes AS
# WITH row_flags AS (
#   SELECT
#     hospitalizations_joined_id,
#     -- Alcohol withdrawal (row-level)
#     CASE WHEN
#          diagnosis_code ILIKE 'F10.130%' OR diagnosis_code ILIKE 'F10.131%' OR
#          diagnosis_code ILIKE 'F10.132%' OR diagnosis_code ILIKE 'F10.139%' OR
#          diagnosis_code ILIKE 'F10.230%' OR diagnosis_code ILIKE 'F10.231%' OR
#          diagnosis_code ILIKE 'F10.232%' OR diagnosis_code ILIKE 'F10.239%' OR
#          diagnosis_code ILIKE 'F10.930%' OR diagnosis_code ILIKE 'F10.931%' OR
#          diagnosis_code ILIKE 'F10.932%'
#     THEN 1 ELSE 0 END AS alcohol_withdrawal_flag,

#     -- Seizure (row-level)
#     CASE WHEN
#          diagnosis_code ILIKE 'R56.9%' OR diagnosis_code ILIKE 'G40.909%'
#     THEN 1 ELSE 0 END AS seizure_flag,

#     -- Increased ICP-related (row-level)
#     CASE WHEN
#          diagnosis_code ILIKE 'G93.2%' OR diagnosis_code ILIKE 'G93.6%' OR
#          diagnosis_code ILIKE 'G93.5%' OR diagnosis_code ILIKE 'H47.11%'
#     THEN 1 ELSE 0 END AS icp_flag
#   FROM '{diag_path}'
# )
# SELECT
#   hospitalizations_joined_id,
#   CASE WHEN MAX(alcohol_withdrawal_flag) = 1 THEN 1 ELSE 0 END AS alcohol_withdrawal_flag,
#   CASE WHEN MAX(seizure_flag)            = 1 THEN 1 ELSE 0 END AS seizure_flag,
#   CASE WHEN MAX(icp_flag)                = 1 THEN 1 ELSE 0 END AS icp_flag
# FROM row_flags
# GROUP BY hospitalizations_joined_id
# ;

# -- View results (order at query time)
# SELECT * FROM diagnosis_codes;
# """)


<h1> Hourly Paired PF Ratio </h1>

In [ ]:
po2_arterial = con.sql(f"""
SELECT
    hospitalizations_joined_id,
    DATE(lab_collect_dttm) AS po2_date,
    HOUR(lab_collect_dttm) AS po2_hour,
    lab_collect_dttm AS hour_ts,
    lab_value_numeric AS po2_arterial,
    reference_unit
FROM '{labs_path}'
WHERE lab_category = 'po2_arterial'
""").df()
po2_arterial

In [ ]:
fio2_set_df = con.sql(f"""
SELECT
    hospitalizations_joined_id,
    recorded_date AS date,
    recorded_hour AS hour,
    MAKE_TIMESTAMP(
        YEAR(recorded_date), MONTH(recorded_date), DAY(recorded_date),
        recorded_hour, 0, 0
    ) AS hour_ts,
    vent_episode_id,
    device_category,
    fio2_set
FROM '{hourly_resp_path}'
WHERE device_category = 'imv'
    AND vent_episode_id = '1'
    AND vent_episode_duration_hours >= '24'
    AND location_category = 'icu'
""").df()
fio2_set_df

In [ ]:
# merge_asof requires sorted data
fio2_set_df = fio2_set_df.sort_values('hour_ts')
po2_arterial = po2_arterial.sort_values('hour_ts')

po2_arterial['hour_ts'] = po2_arterial['hour_ts'].dt.tz_localize(None)

pf_ratio_df = pd.merge_asof(
    fio2_set_df,
    po2_arterial,
    by='hospitalizations_joined_id',
    on='hour_ts',
    direction='nearest',
    tolerance=pd.Timedelta('1 hour')
)

pf_ratio_df['pf_ratio'] = pf_ratio_df['po2_arterial'] / pf_ratio_df['fio2_set']
pf_ratio_df = pf_ratio_df[pf_ratio_df['pf_ratio'].notna()]

pf_ratio_df

In [ ]:
con.register('pf_ratio_df', pf_ratio_df)
con.sql(f"""
CREATE OR REPLACE TEMP VIEW pf_ratio_paired AS
    SELECT
        hospitalizations_joined_id,
        date(date) as date,
        MIN(pf_ratio) AS pf_ratio_paired_min
    FROM pf_ratio_df
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM pf_ratio_paired
""")

<h1> LAPS2 </h1>

<h2> Save vent hospitalizations for LAPS2 export via R function </h2>

In [ ]:
con.sql(f"""
    COPY (
        SELECT
            hospitalizations_joined_id,
            ANY_VALUE(mdm_link_id) AS mdm_link_id,
            ANY_VALUE(mdm_link_id) AS patient_id
        FROM data
        GROUP BY hospitalizations_joined_id
    ) TO 'laps2_hospitalizations.parquet' (FORMAT PARQUET)
""")

<h2> Call laps2 function and create laps2 view </h2>

In [ ]:
!run_laps2.bat

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW laps2 AS
    SELECT DISTINCT ON (hospitalizations_joined_id, recorded_date)
        hospitalizations_joined_id,
        recorded_date as date,
        laps2
    FROM 'laps2_data.parquet';
SELECT * FROM laps2;
""")

<h1> Merge duckdb tables </h1>

In [ ]:
print(con.sql(f"SELECT COUNT(*) FROM data"))
con.sql(f"""
CREATE OR REPLACE TABLE final_df AS
    SELECT
        data.*,
        patient.* EXCLUDE (mdm_link_id),
        height.height_cm,
        hospitalization.* EXCLUDE (hospitalizations_joined_id),
        vent_daily.* EXCLUDE (hospitalizations_joined_id, date),
        prone.prone_position,
        sup.sup,
        ecmo.ecmo,
        ph.ph_min_arterial_or_venous,
        pco2.pco2_min_arterial_or_venous,
        paralytic_meds.paralytic_meds,
        laps2.laps2,
        spo2_min.spo2_min,
        pf_ratio_paired.pf_ratio_paired_min
    FROM data
    LEFT JOIN patient USING (mdm_link_id)
    LEFT JOIN height USING (mdm_link_id)
    LEFT JOIN hospitalization USING (hospitalizations_joined_id)
    LEFT JOIN vent_daily USING (hospitalizations_joined_id, date)
    LEFT JOIN prone USING (hospitalizations_joined_id, date)
    LEFT JOIN sup USING (hospitalizations_joined_id, date)
    LEFT JOIN ecmo USING (hospitalizations_joined_id, date)
    LEFT JOIN ph USING (hospitalizations_joined_id, date)
    LEFT JOIN pco2 USING (hospitalizations_joined_id, date)
    LEFT JOIN paralytic_meds USING (hospitalizations_joined_id, date)
    LEFT JOIN laps2 USING (hospitalizations_joined_id, date)
    LEFT JOIN spo2_min USING (hospitalizations_joined_id, date)
    LEFT JOIN pf_ratio_paired USING (hospitalizations_joined_id, date);
SELECT * FROM final_df
""")
print(con.sql(f"SELECT COUNT(*) FROM final_df"))
# Make sure row numbers stay the same

In [ ]:
data = con.sql(f"SELECT * FROM final_df").df()
data

<h1> BMI </h1>

In [ ]:
weight = con.sql(f"""
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_dttm) AS date,
        vital_value AS weight_kg
    FROM '{vitals_path}'
    WHERE vital_category = 'weight_kg'
        AND vital_value IS NOT NULL
    ORDER BY recorded_dttm
""").df()

data = pd.merge_asof(
    data.sort_values('date'),
    weight.sort_values('date'),
    by='hospitalizations_joined_id',
    on='date',
    direction='nearest',
    allow_exact_matches=True
)

data = data.sort_values(by=['hospitalizations_joined_id', 'date'])
data['bmi'] = round(data['weight_kg'] / (data['height_cm'] / 100) ** 2, 2)
data

<h1> Pandas engineering </h1>

In [ ]:
# Map hospital IDs
hosp_ids = {
    'Woodwinds' : 'Hospital 1',
    'St. Johns' : 'Hospital 2',
    'Southdale' : 'Hospital 3',
    'Riverside' : 'Hospital 4',
    'Ridges' : 'Hospital 5',
    'Ranges/Hibbing' : 'Hospital 6',
    'Northland' : 'Hospital 7',
    'Lakes' : 'Hospital 8',
    'Grand Itasca' : 'delete',
    'UMMC' : 'Hospital Ref',
    'St. Joes' : 'delete',
    'Bethesda' : 'delete'
}
data['hospital_id'] = data['hospital_id'].map(hosp_ids)
data[['hospital_id']]

In [ ]:
# Get mortality
data['mortality_inhospital'] = (data['death_date'].dt.date <= data['discharge_dttm'].dt.date).astype(bool)

In [ ]:
# Check for vent duration of at least 48hr for sup model
data['vent_episode_duration_48hr_min'] = (data['vent_episode_duration_hours'] >= 48).astype(bool)

In [ ]:
# Fillna meds
data['paralytic_meds'] = data['paralytic_meds'].fillna(False)

# Fillna prone
data['prone_position'] = data['prone_position'].fillna(False)

# Fillna sup
data['sup'] = data['sup'].fillna(False)

<h1> Exclusion/Inclusion </h1>

In [ ]:
# Remove tracheostomy rows
print(f'Rows before dropping trach: {len(data)}')
data = data[data['tracheostomy']!= 1].copy()
data['tracheostomy'] = data['tracheostomy'].fillna(False).astype(bool)
print(f'Rows after dropping trach: {len(data)}')

data

In [ ]:
# Remove ECMO days
print(len(data))
data['ecmo'] = data['ecmo'].fillna(False)
data = data[data['ecmo'] == False].copy()
print(len(data))

In [ ]:
## Leftover exclusions
# Make sure 18+
print(f'Starting row n: {len(data)}')

data = data[data['age'] >= 18]
print(f'Age: {len(data)}')

# Remove st. joes and bethesda
data = data[data['hospital_id'] != 'delete']
print(f'Hospital ID: {len(data)}')

# Remove weird stuff
data = data.dropna(subset=['prov_npi_shifted'])
data = data[~data['prov_npi_shifted'].isin(['None', 'none', 'nan'])]
print(f'Provider: {len(data)}')

data

<h1> Missingness</h1>

In [ ]:
print(len(data))
data.isna().sum()

<h1> ARDS </h1>

In [ ]:
!run_ards.bat

In [ ]:
ards_classifier_cohort = pd.read_csv('ards_classifier_cohort.csv', index_col=0)
ards_classifier_cohort["hospitalizations_joined_id"] = ards_classifier_cohort["hospitalization_id"].str.replace(
    r"^([^_]+_[^_]+).*", r"\1", regex=True)
print(len(ards_classifier_cohort))

# some duplicate rows remain - drop
ards_classifier_cohort = ards_classifier_cohort.drop_duplicates()
print(len(ards_classifier_cohort))

# Some duplicate hospitalizations_joined_id > only difference is cardiac_arrest_primary_dx column
# Remove duplicate cardiac_arres_primary_dx
ards_classifier_cohort = ards_classifier_cohort.sort_values('cardiac_arrest_primary_dx', ascending = False)\
    .drop_duplicates('hospitalizations_joined_id')
print(len(ards_classifier_cohort))

# Merge onto data
print(len(data))
data = data.merge(ards_classifier_cohort[['hospitalizations_joined_id', 'cohort_eligible']], on='hospitalizations_joined_id', how='inner')
data['ards_eligible'] = data['cohort_eligible']
data

<h1> Save final dataframe </h1>

In [ ]:
data.to_parquet('LPV_final_data.parquet')